# Detecção de anomalias em transações de cartão de crédito

Neste notebook, analiso o dataset público **Credit Card Fraud Detection**, disponível no Kaggle, para construir um fluxo completo de detecção de fraudes em transações de cartão de crédito.

O objetivo não é produzir um sistema antifraude pronto para produção. O que eu faço aqui é mais delimitado: carregar os dados, entender o desbalanceamento, treinar modelos supervisionados, avaliar os resultados com métricas adequadas e escolher um limiar de classificação a partir das probabilidades estimadas.

A análise usa `pandas`, `numpy`, `scikit-learn` e `plotly`. O modelo principal é um **Random Forest**, e a avaliação é feita principalmente pela **curva Precision-Recall** e pela **Average Precision**, com a curva ROC como leitura complementar.

## Como usar este notebook

Antes de executar tudo, é preciso ter o arquivo `creditcard.csv` disponível localmente. O dataset pode ser baixado em:

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

Por padrão, o código procura o arquivo nestes caminhos:

- `data/creditcard.csv`
- `creditcard.csv`
- `/mnt/data/creditcard.csv`

Se o arquivo estiver em outro lugar, basta alterar a variável `DATA_PATHS` na seção de configuração.

## O que orienta as decisões do projeto

Sigo uma estrutura próxima do CRISP-DM, porque ela ajuda a manter o projeto organizado: primeiro entendo o problema e os dados, depois preparo as variáveis, treino os modelos e avalio os resultados. Bakumenko e Elragal (2022) usam esse tipo de estrutura em um projeto de detecção de anomalias financeiras e mostram que a avaliação precisa estar conectada ao objetivo do problema, não apenas a uma métrica genérica.

O dataset é fortemente desbalanceado: a classe de fraude representa uma fração muito pequena das transações. Por isso, eu não uso acurácia como critério principal. A literatura sobre esse dataset reforça que a **AUPRC / Average Precision** é mais informativa do que a ROC-AUC quando há muitos verdadeiros negativos e poucos casos positivos.

Também trato o resultado do modelo como um **score de risco**. Isso é importante porque, na prática, a decisão não termina no `predict`: preciso escolher um limiar para transformar probabilidades em alertas. Vanini et al. (2023) discutem exatamente essa ligação entre detecção, falsos alertas e gestão de risco.

As técnicas de balanceamento são usadas com cuidado, uma vez que Alamri e Ykhlef (2022) mostram que oversampling, undersampling e abordagens híbridas podem ajudar, mas também podem introduzir problemas como overfitting, perda de informação ou resultados pouco realistas. Como a stack do projeto é limitada ao scikit-learn, eu uso `class_weight` e, como comparação simples, um undersampling feito apenas no conjunto de treino.

Por fim, mantenho as limitações do problema à vista. Dal Pozzolo et al. (2017) destacam que sistemas reais de fraude lidam com concept drift, atraso na confirmação dos rótulos e restrições operacionais de investigação. Este notebook não resolve esses pontos; ele apenas os reconhece para evitar uma conclusão mais forte do que os dados permitem.

## Referências usadas na construção do notebook

- Bakumenko, A.; Elragal, A. **Detecting Anomalies in Financial Data Using Machine Learning Algorithms**. *Systems*, 2022. https://www.mdpi.com/2079-8954/10/5/130
- Jiang, S.; Dong, R.; Wang, J.; Xia, M. **Credit Card Fraud Detection Based on Unsupervised Attentional Anomaly Detection Network**. *Systems*, 2023. https://www.mdpi.com/2079-8954/11/6/305
- Leevy, J. L.; Hancock, J.; Khoshgoftaar, T. M. **Comparative analysis of binary and one-class classification techniques for credit card fraud data**. *Journal of Big Data*, 2023. https://link.springer.com/article/10.1186/s40537-023-00794-5
- Vanini, P.; Rossi, S.; Zvizdic, E.; Domenig, T. **Online payment fraud: from anomaly detection to risk management**. *Financial Innovation*, 2023. https://link.springer.com/article/10.1186/s40854-023-00470-w
- Alamri, M.; Ykhlef, M. **Survey of Credit Card Anomaly and Fraud Detection Using Sampling Techniques**. *Electronics*, 2022. https://www.mdpi.com/2079-9292/11/23/4003
- Dal Pozzolo, A.; Boracchi, G.; Caelen, O.; Alippi, C.; Bontempi, G. **Credit Card Fraud Detection: A Realistic Modeling and a Novel Learning Strategy**. *IEEE Transactions on Neural Networks and Learning Systems*, 2017. https://re.public.polimi.it/bitstream/11311/1044896/1/08038008.pdf

## 1. Configuração inicial

Começamos importando as bibliotecas, fixando uma seed aleatória e deixando os principais parâmetros em um único lugar. Isso facilita reproduzir o experimento e ajustar o notebook sem procurar valores espalhados pelo código.

In [ ]:
import os
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
TARGET = "Class"

DATA_PATHS = [
    Path("data/creditcard.csv"),
    Path("creditcard.csv"),
    Path("/mnt/data/creditcard.csv"),
]

TEST_SIZE = 0.20
VALID_SIZE_FROM_TRAIN = 0.25
TARGET_RECALL = 0.80

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 2. Carregamento dos dados

Nesta etapa, leio o arquivo `creditcard.csv` e verifico se ele tem a estrutura esperada. O dataset contém variáveis anonimizadas por PCA (`V1` a `V28`), além de `Time`, `Amount` e `Class`. Como as variáveis principais foram transformadas, não tento interpretar cada componente como se fosse uma variável de negócio comum.

In [ ]:
def find_data_path(paths):
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Não encontrei o arquivo creditcard.csv. "
        "Baixe o dataset no Kaggle e coloque o arquivo em data/creditcard.csv "
        "ou ajuste a lista DATA_PATHS."
    )

DATA_PATH = find_data_path(DATA_PATHS)
df = pd.read_csv(DATA_PATH)

print(f"Arquivo carregado: {DATA_PATH}")
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]:,}")
df.head()

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(4),
    "unique": df.nunique(),
})

print(f"Duplicatas exatas: {df.duplicated().sum():,}")
summary

## 3. Distribuição da variável-alvo

Aqui eu olho para a proporção entre transações legítimas e fraudulentas. Essa é uma etapa central, porque o problema é definido pelo desbalanceamento. Um modelo pode acertar quase todas as linhas simplesmente prevendo a classe majoritária, então a avaliação precisa olhar diretamente para a classe de fraude.

In [ ]:
class_distribution = (
    df[TARGET]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="quantidade")
)
class_distribution["proporcao"] = class_distribution["quantidade"] / len(df)
class_distribution["proporcao_pct"] = class_distribution["proporcao"].mul(100).round(4)
class_distribution

In [ ]:
fig = px.bar(
    class_distribution,
    x="classe",
    y="quantidade",
    text="proporcao_pct",
    title="Distribuição das classes",
    labels={"classe": "Classe", "quantidade": "Quantidade", "proporcao_pct": "% do total"},
)
fig.update_traces(texttemplate="%{text:.4f}%", textposition="outside")
fig.update_layout(xaxis=dict(tickmode="array", tickvals=[0, 1], ticktext=["Legítima", "Fraude"]))
fig.show()

## 4. Análise exploratória

A exploração é curta e direcionada: observo `Amount`, `Time` e algumas componentes PCA; como as componentes foram anonimizadas, a leitura é estatística: procuro diferenças de distribuição entre as classes, sem atribuir significado de negócio às variáveis `V1` a `V28`.

In [ ]:
edf = df.copy()
edf["log_amount"] = np.log1p(edf["Amount"])
edf["classe_nome"] = np.where(edf[TARGET] == 1, "Fraude", "Legítima")

fig = px.histogram(
    edf,
    x="log_amount",
    color="classe_nome",
    nbins=80,
    histnorm="probability density",
    barmode="overlay",
    opacity=0.65,
    title="Distribuição de Amount em escala logarítmica",
    labels={"log_amount": "log(Amount + 1)", "classe_nome": "Classe"},
)
fig.show()

In [ ]:
amount_summary = (
    df.groupby(TARGET)["Amount"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(4)
)
amount_summary.index = ["Legítima", "Fraude"]
amount_summary

In [ ]:
edf["relative_hour"] = ((edf["Time"] // 3600) % 24).astype(int)

hourly = (
    edf.groupby("relative_hour")
    .agg(
        transacoes=(TARGET, "size"),
        fraudes=(TARGET, "sum"),
    )
    .reset_index()
)
hourly["taxa_fraude_pct"] = 100 * hourly["fraudes"] / hourly["transacoes"]

fig = px.line(
    hourly,
    x="relative_hour",
    y="taxa_fraude_pct",
    markers=True,
    title="Taxa de fraude por hora relativa",
    labels={"relative_hour": "Hora relativa", "taxa_fraude_pct": "Taxa de fraude (%)"},
)
fig.show()

In [ ]:
pca_cols = [col for col in df.columns if col.startswith("V")]
mean_diff = (
    df.groupby(TARGET)[pca_cols]
    .mean()
    .diff()
    .iloc[-1]
    .abs()
    .sort_values(ascending=False)
    .reset_index()
)
mean_diff.columns = ["feature", "abs_mean_difference"]
mean_diff.head(10)

In [ ]:
top_features = mean_diff.head(6)["feature"].tolist()
plot_df = edf[[TARGET, "classe_nome"] + top_features].melt(
    id_vars=[TARGET, "classe_nome"],
    value_vars=top_features,
    var_name="feature",
    value_name="valor",
)

fraud_rows = plot_df[plot_df[TARGET] == 1]
legit_rows = plot_df[plot_df[TARGET] == 0].sample(
    n=min(20000, (plot_df[TARGET] == 0).sum()),
    random_state=RANDOM_STATE,
)
plot_sample = pd.concat([fraud_rows, legit_rows], ignore_index=True)

fig = px.box(
    plot_sample,
    x="feature",
    y="valor",
    color="classe_nome",
    points=False,
    title="Componentes PCA com maior diferença média entre as classes",
    labels={"feature": "Feature", "valor": "Valor", "classe_nome": "Classe"},
)
fig.show()

## 5. Preparação dos dados

Separo `X` e `y`, mantenho uma divisão estratificada e padronizo `Amount` e `Time`. As componentes `V1` a `V28` já vêm de uma transformação PCA, então elas entram sem nova padronização obrigatória.

A divisão fica em treino, validação e teste. O conjunto de validação é usado para comparar modelos e escolher o limiar. O teste fica reservado para a avaliação final.

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f"A coluna alvo {TARGET!r} não foi encontrada no dataframe.")

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALID_SIZE_FROM_TRAIN,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

split_report = pd.DataFrame({
    "conjunto": ["treino", "validação", "teste"],
    "linhas": [len(X_train), len(X_valid), len(X_test)],
    "fraudes": [int(y_train.sum()), int(y_valid.sum()), int(y_test.sum())],
})
split_report["fraude_pct"] = 100 * split_report["fraudes"] / split_report["linhas"]
split_report

In [ ]:
scale_cols = [col for col in ["Amount", "Time"] if col in X.columns]

preprocess = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), scale_cols),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

print("Colunas padronizadas:", scale_cols)
print("Demais colunas seguem como passthrough.")

## 6. Baselines e modelos

O `DummyClassifier` mostra o desempenho de uma solução que não aprende padrões úteis. A regressão logística serve como baseline supervisionado simples. A Random Forest é o modelo principal, usando pesos de classe para lidar com o desbalanceamento. Também incluo um Random Forest treinada com undersampling simples da classe majoritária (esse undersampling é aplicado apenas no conjunto de treino). A validação e o teste permanecem com a distribuição original, porque é assim que o modelo será avaliado de forma mais honesta.

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ("preprocess", preprocess),
        ("model", model),
    ])

models = {
    "Dummy prior": make_pipeline(
        DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
    ),
    "Regressão logística balanceada": make_pipeline(
        LogisticRegression(
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=RANDOM_STATE,
        )
    ),
    "Random Forest balanceada": make_pipeline(
        RandomForestClassifier(
            n_estimators=250,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ),
}

models

In [ ]:
def make_under_sample(X_train, y_train, majority_multiplier=10):
    # Reduzo a classe majoritária apenas no treino para comparar com class_weight.
    train_df = X_train.copy()
    train_df[TARGET] = y_train.values

    minority = train_df[train_df[TARGET] == 1]
    majority = train_df[train_df[TARGET] == 0]

    n_majority = min(len(majority), len(minority) * majority_multiplier)
    majority_down = resample(
        majority,
        replace=False,
        n_samples=n_majority,
        random_state=RANDOM_STATE,
    )

    sampled = (
        pd.concat([minority, majority_down], axis=0)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

    return sampled.drop(columns=[TARGET]), sampled[TARGET].astype(int)

X_train_us, y_train_us = make_under_sample(X_train, y_train, majority_multiplier=10)

print("Treino original:")
print(y_train.value_counts().rename(index={0: "Legítima", 1: "Fraude"}))
print()
print("Treino com undersampling:")
print(y_train_us.value_counts().rename(index={0: "Legítima", 1: "Fraude"}))

In [ ]:
models["Random Forest com undersampling"] = make_pipeline(
    RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
)

## 7. Funções de avaliação

Aqui eu avalio os modelos usando probabilidades, viabilizando calcular ROC-AUC, Average Precision e também testar vários limiares.

In [ ]:
def get_positive_scores(estimator, X_eval):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_eval)[:, 1]
    if hasattr(estimator, "decision_function"):
        scores = estimator.decision_function(X_eval)
        return (scores - scores.min()) / (scores.max() - scores.min())
    raise TypeError("O estimador precisa expor predict_proba ou decision_function.")


def evaluate_scores(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "roc_auc": roc_auc_score(y_true, y_score),
        "average_precision": average_precision_score(y_true, y_score),
        "threshold": threshold,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "alert_rate_pct": 100 * y_pred.mean(),
    }


def build_threshold_table(y_true, y_score, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    rows = []
    for threshold in thresholds:
        row = evaluate_scores(y_true, y_score, threshold=threshold)
        rows.append(row)

    return pd.DataFrame(rows)


def plot_confusion_matrix(y_true, y_score, threshold, title):
    y_pred = (y_score >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    fig = go.Figure(
        data=go.Heatmap(
            z=cm,
            x=["Predita legítima", "Predita fraude"],
            y=["Real legítima", "Real fraude"],
            text=cm,
            texttemplate="%{text}",
            showscale=True,
        )
    )
    fig.update_layout(title=title, xaxis_title="Predição", yaxis_title="Classe real")
    fig.show()

## 8. Treinamento e comparação inicial na validação

Aqui treinamos os modelos no conjunto de treino e comparamos o desempenho na validação. A tabela abaixo ainda usa limiar `0.5` para precision, recall e F1, mas as métricas de ranking — ROC-AUC e Average Precision — já usam as probabilidades completas.

In [ ]:
fitted_models = {}
validation_scores = {}
validation_rows = []

for name, estimator in models.items():
    if name == "Random Forest com undersampling":
        estimator.fit(X_train_us, y_train_us)
    else:
        estimator.fit(X_train, y_train)

    y_score_valid = get_positive_scores(estimator, X_valid)
    validation_scores[name] = y_score_valid
    fitted_models[name] = estimator

    row = evaluate_scores(y_valid, y_score_valid, threshold=0.5)
    row["modelo"] = name
    validation_rows.append(row)

validation_results = (
    pd.DataFrame(validation_rows)
    .loc[:, ["modelo", "roc_auc", "average_precision", "threshold", "precision", "recall", "f1", "fp", "fn", "tp", "alert_rate_pct"]]
    .sort_values("average_precision", ascending=False)
    .reset_index(drop=True)
)

validation_results

## 9. Curva ROC

A curva ROC ajuda a ver como o modelo separa as classes em diferentes limiares. Mantenho-na porque ela é comum na literatura e facilita comparação, mas não tomo a decisão final só por ela. Em dados muito desbalanceados, a presença de muitos verdadeiros negativos pode deixar a ROC-AUC mais otimista do que o problema permite.

In [ ]:
fig = go.Figure()

for name, scores in validation_scores.items():
    fpr, tpr, _ = roc_curve(y_valid, scores)
    auc_value = roc_auc_score(y_valid, scores)
    fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{name} (AUC={auc_value:.4f})",
        )
    )

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Aleatório",
        line=dict(dash="dash"),
    )
)

fig.update_layout(
    title="Curva ROC na validação",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate / Recall",
)
fig.show()

## 10. Curva Precision-Recall

A curva Precision-Recall é a avaliação mais importante deste notebook. Como a fraude é rara, eu quero observar diretamente a relação entre capturar fraudes e gerar falsos alertas. A Average Precision resume essa curva em um único número, mas o gráfico ajuda a enxergar o comportamento dos modelos em vários limiares.

In [ ]:
fig = go.Figure()

base_rate = y_valid.mean()

for name, scores in validation_scores.items():
    precision, recall, _ = precision_recall_curve(y_valid, scores)
    ap_value = average_precision_score(y_valid, scores)
    fig.add_trace(
        go.Scatter(
            x=recall,
            y=precision,
            mode="lines",
            name=f"{name} (AP={ap_value:.4f})",
        )
    )

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[base_rate, base_rate],
        mode="lines",
        name=f"Base rate ({base_rate:.4f})",
        line=dict(dash="dash"),
    )
)

fig.update_layout(
    title="Curva Precision-Recall na validação",
    xaxis_title="Recall",
    yaxis_title="Precision",
)
fig.show()

## 11. Escolha do modelo e do limiar

Eu escolho o modelo principalmente pela Average Precision na validação. Depois, para esse modelo, testo diferentes limiares. O critério abaixo tenta manter um recall mínimo configurado em `TARGET_RECALL`; entre os limiares que atingem esse recall, eu escolho o de maior precision. Se nenhum limiar atingir esse recall, o código usa o maior F1-score na validação. É mister lembrar que, em uma operação real, esse ponto deveria usar custos de investigação, valor da transação, capacidade da equipe e política de risco.

In [ ]:
selected_model_name = validation_results.iloc[0]["modelo"]
selected_model = fitted_models[selected_model_name]
selected_valid_scores = validation_scores[selected_model_name]

threshold_table = build_threshold_table(y_valid, selected_valid_scores)

candidates = threshold_table[threshold_table["recall"] >= TARGET_RECALL]
if len(candidates) > 0:
    selected_threshold_row = candidates.sort_values(["precision", "f1"], ascending=False).iloc[0]
    selection_reason = f"maior precision entre os limiares com recall >= {TARGET_RECALL:.2f}"
else:
    selected_threshold_row = threshold_table.sort_values("f1", ascending=False).iloc[0]
    selection_reason = "maior F1-score, porque nenhum limiar atingiu o recall mínimo definido"

selected_threshold = float(selected_threshold_row["threshold"])

print(f"Modelo selecionado: {selected_model_name}")
print(f"Limiar selecionado: {selected_threshold:.2f}")
print(f"Critério: {selection_reason}")

threshold_table.sort_values("f1", ascending=False).head(10)

In [ ]:
fig = go.Figure()

for metric in ["precision", "recall", "f1"]:
    fig.add_trace(
        go.Scatter(
            x=threshold_table["threshold"],
            y=threshold_table[metric],
            mode="lines",
            name=metric,
        )
    )

fig.add_vline(x=selected_threshold, line_dash="dash")
fig.update_layout(
    title=f"Métricas por limiar — {selected_model_name}",
    xaxis_title="Limiar",
    yaxis_title="Valor da métrica",
)
fig.show()

In [ ]:
fig = go.Figure()

for metric in ["fp", "fn"]:
    fig.add_trace(
        go.Scatter(
            x=threshold_table["threshold"],
            y=threshold_table[metric],
            mode="lines",
            name=metric.upper(),
        )
    )

fig.add_vline(x=selected_threshold, line_dash="dash")
fig.update_layout(
    title=f"Erros por limiar — {selected_model_name}",
    xaxis_title="Limiar",
    yaxis_title="Quantidade",
)
fig.show()

## 12. Avaliação final no teste

Agora eu avalio o modelo escolhido no conjunto de teste, usando o limiar definido na validação, evitando ajustar o limiar diretamente no teste.

In [ ]:
test_scores = get_positive_scores(selected_model, X_test)
test_metrics = evaluate_scores(y_test, test_scores, threshold=selected_threshold)

test_results = pd.DataFrame([{**{"modelo": selected_model_name}, **test_metrics}])
test_results = test_results.loc[:, [
    "modelo", "roc_auc", "average_precision", "threshold",
    "precision", "recall", "f1", "tn", "fp", "fn", "tp", "alert_rate_pct"
]]
test_results

In [ ]:
y_test_pred = (test_scores >= selected_threshold).astype(int)
print(classification_report(y_test, y_test_pred, target_names=["Legítima", "Fraude"], zero_division=0))

In [ ]:
plot_confusion_matrix(
    y_test,
    test_scores,
    threshold=selected_threshold,
    title=f"Matriz de confusão no teste — {selected_model_name}",
)

## 13. Comparação final dos modelos no teste

A escolha do modelo foi feita na validação, mas ainda assim comparo todos os modelos no teste com o limiar `0.5` para ter uma leitura geral.

Essa tabela não substitui a decisão anterior; ela só ajuda a entender como os modelos se comportam em dados não usados no treinamento.

In [ ]:
test_rows = []
for name, estimator in fitted_models.items():
    scores = get_positive_scores(estimator, X_test)
    row = evaluate_scores(y_test, scores, threshold=0.5)
    row["modelo"] = name
    test_rows.append(row)

all_test_results = (
    pd.DataFrame(test_rows)
    .loc[:, ["modelo", "roc_auc", "average_precision", "threshold", "precision", "recall", "f1", "fp", "fn", "tp", "alert_rate_pct"]]
    .sort_values("average_precision", ascending=False)
    .reset_index(drop=True)
)
all_test_results

## 14. Importância das variáveis

Se o modelo selecionado for uma Random Forest, eu extraio as importâncias das variáveis. Essa análise ajuda a entender quais colunas mais contribuíram para as decisões do modelo.

In [ ]:
def get_rf_model_for_importance(fitted_models, selected_model_name):
    selected = fitted_models[selected_model_name]
    if hasattr(selected.named_steps["model"], "feature_importances_"):
        return selected_model_name, selected

    for name, model in fitted_models.items():
        if hasattr(model.named_steps["model"], "feature_importances_"):
            return name, model

    return None, None

importance_model_name, importance_model = get_rf_model_for_importance(fitted_models, selected_model_name)

if importance_model is None:
    print("Nenhum modelo com feature_importances_ foi encontrado.")
else:
    feature_names = importance_model.named_steps["preprocess"].get_feature_names_out()
    importances = importance_model.named_steps["model"].feature_importances_

    importance_df = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    display(importance_df.head(20))

    fig = px.bar(
        importance_df.head(20).sort_values("importance"),
        x="importance",
        y="feature",
        orientation="h",
        title=f"Top 20 importâncias — {importance_model_name}",
        labels={"importance": "Importância", "feature": "Feature"},
    )
    fig.show()

## 15. Leitura dos resultados

O mais importante desta etapa é não olhar apenas para uma métrica agregada, mas verificar quantas fraudes foram capturadas, quantas ficaram de fora e quantas transações legítimas viraram alerta.

In [ ]:
summary_line = test_results.iloc[0]

print(f"Modelo final: {summary_line['modelo']}")
print(f"Limiar usado: {summary_line['threshold']:.2f}")
print(f"ROC-AUC no teste: {summary_line['roc_auc']:.4f}")
print(f"Average Precision no teste: {summary_line['average_precision']:.4f}")
print(f"Precision no limiar escolhido: {summary_line['precision']:.4f}")
print(f"Recall no limiar escolhido: {summary_line['recall']:.4f}")
print(f"F1 no limiar escolhido: {summary_line['f1']:.4f}")
print(f"Falsos positivos: {int(summary_line['fp'])}")
print(f"Falsos negativos: {int(summary_line['fn'])}")
print(f"Fraudes detectadas: {int(summary_line['tp'])}")
print(f"Taxa de alertas: {summary_line['alert_rate_pct']:.4f}%")

## 16. Limitações

Este notebook usa um dataset público, estático e anonimizado. Isso torna o experimento controlado, mas também limita o alcance das conclusões. Em um cenário real, precisariamos lidar com mudança no comportamento dos clientes e golpistas, atraso na confirmação dos rótulos e capacidade limitada de investigar alertas.

Também não há variáveis de contexto valiosas, como histórico do cliente, estabelecimento, país, dispositivo ou sequência de compras. Esses dados costumam ser importantes em sistemas reais, mas não estão disponíveis aqui. Por isso, a análise mostra como construir e avaliar um modelo com cuidado, não como implantar uma solução completa de prevenção a fraude.

## 17. Conclusão

Embora objetivo e particular o projeto acima demonstra um fluxo completo para detecção de fraudes de cartões de crédito. Nele, verificamos a estrutura dos dados e confirmamos o desbalanceamento da classe de fraude; depois fizemos uma exploração breve, preparamos os dados com divisão estratificada, treinamos baselines e comparamos modelos supervisionados.

A avaliação foi guiada principalmente pela Average Precision e pela curva Precision-Recall, porque a classe de fraude é rara e o volume de falsos positivos importa. A ROC-AUC foi mantida como métrica complementar, mas não como critério principal.

O modelo final foi escolhido na validação, e o limiar foi definido antes da avaliação no teste. Com isso, a conclusão fica mais controlada: conseguimos dizer como o modelo se comportou neste dataset, com esta divisão e este critério de limiar, sem cometer o equívoco de tratar resultado como garantia de desempenho em produção.

Como próximos passos, eu testaria modelos adicionais, validação temporal, calibração de probabilidades e uma função de custo mais próxima do negócio. Também investigaria estratégias para lidar com drift e atraso de rótulos, que são problemas centrais em sistemas reais de fraude.